# 🛡️ Data Quality Monitoring
## Catalog: `data_mesh_hub.platform` | Owner: Data Platform Team

In a Data Mesh, **quality is everyone's responsibility** but the **Platform Team provides the tooling**. This notebook runs quality checks across ALL data products and logs results to `data_mesh_hub.platform.data_quality_log`.

**Characteristics demonstrated:**
- ✅ Quality Checks — Completeness, validity, uniqueness, freshness, referential integrity
- ✅ Reliability / Freshness — SLA compliance monitoring
- ✅ Cross-Domain Governance — Checks span all 3 catalogs

**Quality Check Types:**
| Type | What It Checks |
|------|---------------|
| Completeness | % of non-null values in key fields |
| Validity | Values within expected ranges/sets |
| Uniqueness | No duplicate primary keys |
| Freshness | Data refreshed within SLA window |
| Referential Integrity | Cross-catalog FK references valid |

## ▶️ Run Quality Checks
Execute checks across all 3 data products and log results.

In [0]:
%sql
-- ============================================================
-- Execute quality checks across ALL data products
-- Results logged to data_mesh_hub.platform.data_quality_log
-- ============================================================
INSERT INTO data_mesh_hub.platform.data_quality_log

-- ADOIT PRODUCT CHECKS
SELECT 'QC-' || CAST(UNIX_TIMESTAMP() AS STRING) || '-001', current_timestamp(), 
    'Application Landscape', 'EA', 'adoit_product.gold.application_landscape',
    'completeness_app_owner', 'completeness', 'All applications should have an owner',
    (SELECT COUNT(CASE WHEN app_owner IS NOT NULL THEN 1 END) * 100.0 / COUNT(*) FROM adoit_product.gold.application_landscape) >= 95,
    '>= 95%', CONCAT(CAST((SELECT ROUND(COUNT(CASE WHEN app_owner IS NOT NULL THEN 1 END) * 100.0 / COUNT(*), 1) FROM adoit_product.gold.application_landscape) AS STRING), '%'), 'Warning'

UNION ALL SELECT 'QC-' || CAST(UNIX_TIMESTAMP() AS STRING) || '-002', current_timestamp(),
    'Application Landscape', 'EA', 'adoit_product.gold.application_landscape',
    'uniqueness_app_id', 'uniqueness', 'app_id must be unique',
    (SELECT COUNT(*) = COUNT(DISTINCT app_id) FROM adoit_product.gold.application_landscape),
    '0 duplicates', CONCAT(CAST((SELECT COUNT(*) - COUNT(DISTINCT app_id) FROM adoit_product.gold.application_landscape) AS STRING), ' duplicates'), 'Critical'

UNION ALL SELECT 'QC-' || CAST(UNIX_TIMESTAMP() AS STRING) || '-003', current_timestamp(),
    'Application Landscape', 'EA', 'adoit_product.gold.application_landscape',
    'validity_criticality', 'validity', 'criticality must be Critical/High/Medium/Low',
    (SELECT COUNT(CASE WHEN criticality NOT IN ('Critical','High','Medium','Low') THEN 1 END) FROM adoit_product.gold.application_landscape) = 0,
    '0 invalid', CAST((SELECT COUNT(CASE WHEN criticality NOT IN ('Critical','High','Medium','Low') THEN 1 END) FROM adoit_product.gold.application_landscape) AS STRING), 'Critical'

UNION ALL SELECT 'QC-' || CAST(UNIX_TIMESTAMP() AS STRING) || '-004', current_timestamp(),
    'Application Landscape', 'EA', 'adoit_product.gold.application_landscape',
    'freshness_24h', 'freshness', 'Data refreshed within 24 hours',
    (SELECT TIMESTAMPDIFF(HOUR, MAX(product_generated_at), current_timestamp()) FROM adoit_product.gold.application_landscape) <= 24,
    '<= 24 hours', CONCAT(CAST((SELECT TIMESTAMPDIFF(HOUR, MAX(product_generated_at), current_timestamp()) FROM adoit_product.gold.application_landscape) AS STRING), ' hours'), 'Critical'

-- SNOW PRODUCT CHECKS
UNION ALL SELECT 'QC-' || CAST(UNIX_TIMESTAMP() AS STRING) || '-005', current_timestamp(),
    'Service Health', 'ITSM', 'snow_product.gold.service_health',
    'validity_sla_range', 'validity', 'SLA compliance must be 0-100%',
    (SELECT COUNT(*) FROM snow_product.gold.service_health WHERE sla_compliance_pct < 0 OR sla_compliance_pct > 100) = 0,
    '0 out-of-range', CAST((SELECT COUNT(*) FROM snow_product.gold.service_health WHERE sla_compliance_pct < 0 OR sla_compliance_pct > 100) AS STRING), 'Critical'

UNION ALL SELECT 'QC-' || CAST(UNIX_TIMESTAMP() AS STRING) || '-006', current_timestamp(),
    'Service Health', 'ITSM', 'snow_product.gold.service_health',
    'completeness_coverage', 'completeness', 'All apps with incidents should have health record',
    (SELECT COUNT(DISTINCT affected_application_id) FROM snow_product.silver.incidents WHERE affected_application_id IS NOT NULL) <= (SELECT COUNT(*) FROM snow_product.gold.service_health),
    'All covered', CONCAT(CAST((SELECT COUNT(*) FROM snow_product.gold.service_health) AS STRING), ' of ', CAST((SELECT COUNT(DISTINCT affected_application_id) FROM snow_product.silver.incidents WHERE affected_application_id IS NOT NULL) AS STRING)), 'Critical'

-- CROSS-DOMAIN CHECKS
UNION ALL SELECT 'QC-' || CAST(UNIX_TIMESTAMP() AS STRING) || '-007', current_timestamp(),
    'Application Risk Profile', 'Cross-Domain', 'data_mesh_hub.cross_domain.application_risk_profile',
    'referential_integrity', 'referential_integrity', 'All app_ids must exist in adoit_product',
    (SELECT COUNT(*) FROM data_mesh_hub.cross_domain.application_risk_profile r LEFT JOIN adoit_product.gold.application_landscape a ON r.app_id = a.app_id WHERE a.app_id IS NULL) = 0,
    '0 orphans', CONCAT(CAST((SELECT COUNT(*) FROM data_mesh_hub.cross_domain.application_risk_profile r LEFT JOIN adoit_product.gold.application_landscape a ON r.app_id = a.app_id WHERE a.app_id IS NULL) AS STRING), ' orphans'), 'Critical'

UNION ALL SELECT 'QC-' || CAST(UNIX_TIMESTAMP() AS STRING) || '-008', current_timestamp(),
    'Application Risk Profile', 'Cross-Domain', 'data_mesh_hub.cross_domain.application_risk_profile',
    'completeness_risk_factors', 'completeness', 'High/Critical risk apps should have risk factors',
    (SELECT COUNT(CASE WHEN composite_risk_score IN ('CRITICAL','HIGH') AND (risk_factors IS NULL OR risk_factors = '') THEN 1 END) FROM data_mesh_hub.cross_domain.application_risk_profile) = 0,
    '0 missing', CAST((SELECT COUNT(CASE WHEN composite_risk_score IN ('CRITICAL','HIGH') AND (risk_factors IS NULL OR risk_factors = '') THEN 1 END) FROM data_mesh_hub.cross_domain.application_risk_profile) AS STRING), 'Warning';

## 📊 Quality Dashboard
View the latest quality check results across all data products.

In [0]:
%sql
-- Quality summary by data product
SELECT 
    data_product,
    domain,
    COUNT(*) AS total_checks,
    SUM(CASE WHEN passed THEN 1 ELSE 0 END) AS passed,
    SUM(CASE WHEN NOT passed THEN 1 ELSE 0 END) AS failed,
    ROUND(SUM(CASE WHEN passed THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 0) AS pass_rate_pct
FROM data_mesh_hub.platform.data_quality_log
WHERE check_timestamp >= current_timestamp() - INTERVAL 24 HOURS
GROUP BY data_product, domain
ORDER BY pass_rate_pct;

In [0]:
%sql
-- Detailed latest check results (most recent run)
SELECT 
    data_product,
    check_name,
    check_type,
    CASE WHEN passed THEN '✅ PASS' ELSE '❌ FAIL' END AS status,
    expected_value,
    actual_value,
    severity
FROM data_mesh_hub.platform.data_quality_log
WHERE check_timestamp >= current_timestamp() - INTERVAL 24 HOURS
ORDER BY passed ASC, severity DESC;

In [0]:
%sql
-- Freshness monitoring: how old is each data product?
SELECT 
    'adoit_product.gold.application_landscape' AS data_product,
    'EA' AS domain,
    'Daily by 08:00 UTC' AS sla,
    TIMESTAMPDIFF(MINUTE, MAX(product_generated_at), current_timestamp()) AS age_minutes,
    CASE WHEN TIMESTAMPDIFF(HOUR, MAX(product_generated_at), current_timestamp()) <= 24 THEN '✅ Fresh' ELSE '❌ Stale' END AS freshness
FROM adoit_product.gold.application_landscape
UNION ALL
SELECT 
    'snow_product.gold.service_health',
    'ITSM',
    'Hourly',
    TIMESTAMPDIFF(MINUTE, MAX(product_generated_at), current_timestamp()),
    CASE WHEN TIMESTAMPDIFF(MINUTE, MAX(product_generated_at), current_timestamp()) <= 60 THEN '✅ Fresh' ELSE '⚠️ Check SLA' END
FROM snow_product.gold.service_health
UNION ALL
SELECT 
    'data_mesh_hub.cross_domain.application_risk_profile',
    'Cross-Domain',
    'Daily by 09:00 UTC',
    TIMESTAMPDIFF(MINUTE, MAX(product_generated_at), current_timestamp()),
    CASE WHEN TIMESTAMPDIFF(HOUR, MAX(product_generated_at), current_timestamp()) <= 24 THEN '✅ Fresh' ELSE '❌ Stale' END
FROM data_mesh_hub.cross_domain.application_risk_profile;

## 📈 Quality Trend (Historical)
Over time, this log accumulates and enables trend analysis for data product health.

In [0]:
%sql
-- Historical quality checks (all time)
SELECT 
    DATE(check_timestamp) AS check_date,
    data_product,
    COUNT(*) AS checks_run,
    SUM(CASE WHEN passed THEN 1 ELSE 0 END) AS passed,
    SUM(CASE WHEN NOT passed THEN 1 ELSE 0 END) AS failed
FROM data_mesh_hub.platform.data_quality_log
GROUP BY DATE(check_timestamp), data_product
ORDER BY check_date DESC, data_product;